# Generate Summaries with Fine-tuned Mistral-7B

Loads the QLoRA adapter and generates summaries for 200 products.
Download the results and run LLM-as-Judge + A/B test locally.

**Upload these files:**
1. `summary_training_pairs.jsonl` (from data/processed/)
2. `mistral-qlora-adapter.zip` (from models/)

In [1]:
# Cell 1: Mount Drive + Install deps
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers peft bitsandbytes accelerate

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [2]:
# Cell 2: Upload files (first run only)
import os
from google.colab import files

# Upload if not on Drive already
if not os.path.exists('/content/drive/MyDrive/summary_training_pairs.jsonl'):
    uploaded = files.upload()
    !cp summary_training_pairs.jsonl /content/drive/MyDrive/
else:
    !cp /content/drive/MyDrive/summary_training_pairs.jsonl .
    print('Loaded data from Drive')

# Upload adapter if not on Drive
if not os.path.exists('/content/drive/MyDrive/mistral-qlora-adapter'):
    uploaded = files.upload()  # Upload mistral-qlora-adapter.zip
    !unzip -q mistral-qlora-adapter.zip
    !cp -r mistral-qlora-adapter /content/drive/MyDrive/
else:
    !cp -r /content/drive/MyDrive/mistral-qlora-adapter .
    print('Loaded adapter from Drive')

Loaded data from Drive
Loaded adapter from Drive


In [3]:
# Cell 3: Load model + adapter
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtypae=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(base_model, "./mistral-qlora-adapter")
model.eval()
print("Model + adapter loaded")

GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model + adapter loaded


In [4]:
# Cell 4: Load data
pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))

# Use first 200 for evaluation
pairs = pairs[:200]
print(f"Generating summaries for {len(pairs)} products")

Generating summaries for 200 products


In [5]:
# Cell 5: Generate summaries (saves to Drive)
SAVE_PATH = '/content/drive/MyDrive/finetuned_summaries.jsonl'

# Resume support
results = []
start_from = 0
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH) as f:
        results = [json.loads(line) for line in f]
    start_from = len(results)
    print(f"Resuming from {start_from}/{len(pairs)}")

def generate_summary(model, tokenizer, reviews_input):
    prompt = (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{reviews_input[:2000]}\n[/INST]\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated.split('[/INST]')[-1].strip()
    return response

for i in range(start_from, len(pairs)):
    pair = pairs[i]
    summary = generate_summary(model, tokenizer, pair['input'])
    results.append({
        'asin': pair['asin'],
        'product_title': pair.get('product_title', ''),
        'input': pair['input'],
        'summary': summary,
    })
    if (i + 1) % 10 == 0:
        # Save to Drive every 10
        with open(SAVE_PATH, 'w') as f:
            for r in results:
                f.write(json.dumps(r) + '\n')
        print(f"Generated {i + 1} / {len(pairs)} (saved to Drive)")

# Final save
with open(SAVE_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f"Done! {len(results)} summaries saved to Drive")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generated 10 / 200 (saved to Drive)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generated 20 / 200 (saved to Drive)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Generated 180 / 200 (saved to Drive)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generated 190 / 200 (saved to Drive)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generated 200 / 200 (saved to Drive)
Done! 200 summaries saved to Drive


In [ ]:
# Cell 7: Download results
!cp /content/drive/MyDrive/finetuned_summaries.jsonl .
files.download('finetuned_summaries.jsonl')
print("Save to data/processed/finetuned_summaries.jsonl")